# Cyberbullying Detection — TweetEval Dataset (5 classes)
## Original BERT (CLS) vs. BERT + Attention Pooling (Final Model)
Same architecture, same deterministic pipeline as the IEEE DataPort and Kaggle notebooks: identical seed, identical split logic, no `langdetect` (removed for reproducibility), `torch.backends.cudnn.deterministic = True`, `EPOCHS=10` with early stopping (patience=3) on validation loss.

**Only two things change vs. the other two notebooks:**
- Data loading (TweetEval CSV: `text` / `label`, 5 classes)
- `n_output = 5` (same as IEEE DataPort, different from Kaggle's 6)

McNemar's test is included at the end this time — added as a standard check going forward, to confirm whether any accuracy difference between the two models is statistically real or just noise.

## 1. Setup

In [ ]:
!pip install -q emoji contractions imbalanced-learn codecarbon
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import string
import time
import random

import emoji
import contractions
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import RandomOverSampler

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertModel, BertTokenizer
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

seed_value = 2042
random.seed(seed_value)
np.random.seed(seed_value)
torch.manual_seed(seed_value)
torch.cuda.manual_seed_all(seed_value)

# Force fully deterministic GPU behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

sns.set_style("whitegrid")
plt.rc("figure", autolayout=True)
plt.rc("axes", labelweight="bold", labelsize="large", titleweight="bold", titlepad=10)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2. Load TweetEval dataset
Columns `text` / `label`, 5 classes: `hate`, `offensive`, `religion`, `spiritual`, `emotion`. Adjust `base_path` to your Kaggle input directory.

In [ ]:
base_path = '/kaggle/input/tweeteval-cyberbullying/'

df = pd.read_csv(base_path + 'Tweeteval.csv', encoding='latin1')
df = df.rename(columns={'label': 'sentiment'})
df = df[~df.duplicated()]
print(df.shape)
print(df['sentiment'].value_counts())

## 3. Tweet text deep cleaning (identical pipeline to IEEE DataPort/Kaggle — no langdetect, for reproducibility)

In [ ]:
def strip_emoji(text):
    if not isinstance(text, str):
        text = str(text)
    emoji_pattern = re.compile(
        "[\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FA6F"
        "\U0001FA70-\U0001FAFF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

def strip_all_entities(text):
    if not isinstance(text, str):
        text = str(text)
    text = re.sub(r'\r|\n', ' ', text.lower())
    text = re.sub(r"(?:\@|https?\://)\S+", "", text)
    text = re.sub(r'[^\x00-\x7f]', '', text)
    table = str.maketrans('', '', string.punctuation)
    text = text.translate(table)
    text = ' '.join(word for word in text.split() if word not in stop_words)
    return text

def clean_hashtags(tweet):
    if not isinstance(tweet, str):
        tweet = str(tweet)
    new_tweet = re.sub(r'(\s+#[\w-]+)+\s*$', '', tweet).strip()
    new_tweet = re.sub(r'#([\w-]+)', r'\1', new_tweet).strip()
    return new_tweet

def filter_chars(text):
    if not isinstance(text, str):
        text = str(text)
    return ' '.join('' if ('$' in word) or ('&' in word) else word for word in text.split())

def remove_mult_spaces(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(r"\s\s+", " ", text)

def expand_contractions(text):
    if not isinstance(text, str):
        text = str(text)
    return contractions.fix(text)

def remove_numbers(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(r'\d+', '', text)

def lemmatize(text):
    if not isinstance(text, str):
        text = str(text)
    words = word_tokenize(text)
    return ' '.join(lemmatizer.lemmatize(w) for w in words)

def remove_short_words(text, min_len=2):
    if not isinstance(text, str):
        text = str(text)
    return ' '.join(w for w in text.split() if len(w) >= min_len)

def replace_elongated_words(text):
    if not isinstance(text, str):
        text = str(text)
    regex_pattern = r'\b(\w+)((\w)\3{2,})(\w*)\b'
    return re.sub(regex_pattern, r'\1\3\4', text)

def remove_repeated_punctuation(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(r'[\?\.\!]+(?=[\?\.\!])', '', text)

def remove_extra_whitespace(text):
    if not isinstance(text, str):
        text = str(text)
    return ' '.join(text.split())

def remove_url_shorteners(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(
        r'(?:http[s]?://)?(?:www\.)?(?:bit\.ly|goo\.gl|t\.co|tinyurl\.com|tr\.im|is\.gd|'
        r'cli\.gs|u\.nu|url\.ie|tiny\.cc|alturl\.com|ow\.ly|bit\.do|adoro\.to)\S+', '', text)

def remove_spaces_tweets(tweet):
    if not isinstance(tweet, str):
        tweet = str(tweet)
    return tweet.strip()

def remove_short_tweets(tweet, min_words=3):
    if not isinstance(tweet, str):
        tweet = str(tweet)
    words = tweet.split()
    return tweet if len(words) >= min_words else ""

def clean_tweet(tweet):
    if not isinstance(tweet, str):
        tweet = str(tweet)
    tweet = strip_emoji(tweet)
    tweet = expand_contractions(tweet)
    tweet = strip_all_entities(tweet)
    tweet = clean_hashtags(tweet)
    tweet = filter_chars(tweet)
    tweet = remove_mult_spaces(tweet)
    tweet = remove_numbers(tweet)
    tweet = lemmatize(tweet)
    tweet = remove_short_words(tweet)
    tweet = replace_elongated_words(tweet)
    tweet = remove_repeated_punctuation(tweet)
    tweet = remove_extra_whitespace(tweet)
    tweet = remove_url_shorteners(tweet)
    tweet = remove_spaces_tweets(tweet)
    tweet = ' '.join(tweet.split())
    return tweet

In [ ]:
df['text_clean'] = [clean_tweet(t) for t in df['text']]
print(f'{int(df["text_clean"].duplicated().sum())} duplicated cleaned tweets will be removed.')
df.drop_duplicates('text_clean', inplace=True)
df = df[df['text_clean'].str.len() > 0]
print(df['sentiment'].value_counts())

In [ ]:
sentiment = ["hate", "offensive", "religion", "spiritual", "emotion"]

df['text_len'] = [len(t.split()) for t in df['text_clean']]
df = df[df['text_len'] < df['text_len'].quantile(0.995)]

label_map = {name: i for i, name in enumerate(sentiment)}
df['sentiment'] = df['sentiment'].replace(label_map)
print(f"Final dataset size: {len(df)}")
print(df['sentiment'].value_counts())

## 4. Train / Validation / Test split (same logic as IEEE DataPort — no leakage)

In [ ]:
X = df['text_clean'].values
y = df['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=seed_value)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=seed_value)

print(f"Train: {len(X_train)} | Val: {len(X_valid)} | Test: {len(X_test)}")

## 5. Oversampling (training set only)
TweetEval has moderate imbalance — `emotion` (~5,052 rows) and `spiritual` (~9,442) are notably smaller than `offensive` (~14,100). Oversampling will have a real, visible effect here, more than on Kaggle but less extreme than IEEE DataPort's imbalance.

In [ ]:
ros = RandomOverSampler(random_state=seed_value)
X_train_res, y_train_res = ros.fit_resample(
    np.array(X_train).reshape(-1, 1), np.array(y_train).reshape(-1, 1))

X_train = X_train_res.flatten()
y_train = y_train_res.flatten()

(unique, counts) = np.unique(y_train, return_counts=True)
print("Class balance after oversampling:")
print(np.asarray((unique, counts)).T)

## 6. BERT Tokenization

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)
MAX_LEN = 128

def bert_tokenizer(data):
    input_ids, attention_masks = [], []
    for sent in data:
        encoded_sent = tokenizer(
            sent, add_special_tokens=True, max_length=MAX_LEN,
            padding='max_length', truncation=True, return_attention_mask=True
        )
        input_ids.append(encoded_sent['input_ids'])
        attention_masks.append(encoded_sent['attention_mask'])
    return torch.tensor(input_ids), torch.tensor(attention_masks)

train_inputs, train_masks = bert_tokenizer(X_train)
val_inputs, val_masks = bert_tokenizer(X_valid)
test_inputs, test_masks = bert_tokenizer(X_test)

## 7. DataLoaders

In [ ]:
train_labels = torch.tensor(y_train, dtype=torch.long)
val_labels = torch.tensor(y_valid, dtype=torch.long)
test_labels = torch.tensor(y_test, dtype=torch.long)

batch_size = 32

train_data = TensorDataset(train_inputs, train_masks, train_labels)
train_dataloader = DataLoader(train_data, sampler=RandomSampler(train_data), batch_size=batch_size)

val_data = TensorDataset(val_inputs, val_masks, val_labels)
val_dataloader = DataLoader(val_data, sampler=SequentialSampler(val_data), batch_size=batch_size)

test_data = TensorDataset(test_inputs, test_masks, test_labels)
test_dataloader = DataLoader(test_data, sampler=SequentialSampler(test_data), batch_size=batch_size)

print(len(next(iter(train_dataloader))), len(next(iter(val_dataloader))), len(next(iter(test_dataloader))))

## 8. Model definitions — CLS baseline + Attention Pooling (final model)
Identical architectures to the IEEE DataPort notebooks, only `n_output = 6`.

In [ ]:
N_OUTPUT = 5

class Bert_Classifier_CLS(nn.Module):
    """Baseline — Original BERT, CLS-token pooling."""
    def __init__(self, freeze_bert=False):
        super().__init__()
        n_hidden = 50
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.classifier = nn.Sequential(
            nn.Linear(768, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, N_OUTPUT)
        )
        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_vec = outputs[0][:, 0, :]
        return self.classifier(cls_vec)


class Bert_Classifier_AttentionPool(nn.Module):
    """Final model — BERT + Attention Pooling (replaces CLS-only)."""
    def __init__(self, freeze_bert=False):
        super().__init__()
        n_hidden = 128
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.attention_weights = nn.Sequential(
            nn.Linear(768, 128), nn.Tanh(), nn.Linear(128, 1)
        )
        self.classifier = nn.Sequential(
            nn.Linear(768, n_hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(n_hidden, N_OUTPUT)
        )
        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs[0]
        attn_scores = self.attention_weights(last_hidden_state).squeeze(-1)
        attn_scores = attn_scores.masked_fill(attention_mask == 0, -1e9)
        attn_probs = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled_output = torch.sum(last_hidden_state * attn_probs, dim=1)
        return self.classifier(pooled_output)

## 9. Shared training loop (with early stopping) — used by both models

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def initialize_model(model_class, epochs=10):
    model = model_class(freeze_bert=False)
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=5e-5, eps=1e-8)
    total_steps = len(train_dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
    return model, optimizer, scheduler


def bert_train(model, optimizer, scheduler, train_dataloader, val_dataloader, epochs=10, patience=3, name=""):
    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0

    print(f"Start training: {name}\n")
    for epoch_i in range(epochs):
        t0_epoch = time.time()
        total_loss = 0
        model.train()

        for step, batch in enumerate(train_dataloader):
            b_input_ids, b_attn_mask, b_labels = tuple(t.to(device) for t in batch)
            model.zero_grad()
            logits = model(b_input_ids, b_attn_mask)
            loss = loss_fn(logits, b_labels)
            total_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        avg_train_loss = total_loss / len(train_dataloader)

        model.eval()
        val_accuracy, val_loss = [], []
        for batch in val_dataloader:
            b_input_ids, b_attn_mask, b_labels = tuple(t.to(device) for t in batch)
            with torch.no_grad():
                logits = model(b_input_ids, b_attn_mask)
            loss = loss_fn(logits, b_labels)
            val_loss.append(loss.item())
            preds = torch.argmax(logits, dim=1).flatten()
            val_accuracy.append((preds == b_labels).cpu().numpy().mean() * 100)

        val_loss = np.mean(val_loss)
        val_accuracy = np.mean(val_accuracy)
        elapsed = time.time() - t0_epoch
        print(f"Epoch {epoch_i+1:>2} | train loss {avg_train_loss:.4f} | "
              f"val loss {val_loss:.4f} | val acc {val_accuracy:.2f}% | {elapsed:.1f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch_i+1} (no val improvement for {patience} epochs).")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Restored best checkpoint (val loss = {best_val_loss:.6f}).")
    print(f"Training complete: {name}\n{'='*60}\n")
    return model


def evaluate_on_test(model, test_dataloader, y_test, name):
    model.eval()
    preds_list = []
    for batch in test_dataloader:
        b_input_ids, b_attn_mask, _ = tuple(t.to(device) for t in batch)
        with torch.no_grad():
            logits = model(b_input_ids, b_attn_mask)
        preds_list.extend(torch.argmax(logits, dim=1).cpu().numpy())

    acc = accuracy_score(y_test, preds_list)
    print(f"\n[{name}] TEST ACCURACY: {acc:.4f}\n")
    print(classification_report(y_test, preds_list, target_names=sentiment, digits=3))
    return preds_list, acc

## 10. Run both variants
**Runtime note:** TweetEval is ~24x larger than IEEE DataPort (52K vs 2.1K rows), similar in scale to Kaggle. Budget your Kaggle GPU session time accordingly. Early stopping will help cut this short once the model converges.

In [ ]:
EPOCHS = 10
results = {}
predictions = {}

for model_class, name in [
    (Bert_Classifier_CLS, "Original BERT (CLS-only baseline)"),
    (Bert_Classifier_AttentionPool, "BERT + Attention Pooling (Final Model)"),
]:
    model, optimizer, scheduler = initialize_model(model_class, epochs=EPOCHS)
    model = bert_train(model, optimizer, scheduler, train_dataloader, val_dataloader,
                        epochs=EPOCHS, patience=3, name=name)
    preds, acc = evaluate_on_test(model, test_dataloader, y_test, name)
    results[name] = acc
    predictions[name] = preds

## 11. Summary comparison table

In [ ]:
summary = pd.DataFrame({
    'Model': list(results.keys()),
    'Test Accuracy': [f"{v:.4f}" for v in results.values()]
})
print(summary.to_string(index=False))

In [ ]:
def conf_matrix(y, y_pred, title, labels):
    fig, ax = plt.subplots(figsize=(8, 8))
    sns.heatmap(confusion_matrix(y, y_pred), annot=True, cmap="Purples", fmt='g',
                cbar=False, annot_kws={"size": 14}, xticklabels=labels, yticklabels=labels, ax=ax)
    plt.title(title, fontsize=16)
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.show()

for name, preds in predictions.items():
    conf_matrix(y_test, preds, f'{name}\nConfusion Matrix (TweetEval)', sentiment)

## 12. McNemar's Test — is Attention Pooling statistically better, or is the difference just noise?
This test compares the two models' predictions on the exact same test examples, focusing only on the cases where they disagree.

In [ ]:
!pip install -q statsmodels

from statsmodels.stats.contingency_tables import mcnemar

baseline_name = "Original BERT (CLS-only baseline)"
final_name = "BERT + Attention Pooling (Final Model)"

preds_baseline = np.array(predictions[baseline_name])
preds_final = np.array(predictions[final_name])
y_true = np.array(y_test)

correct_baseline = (preds_baseline == y_true)
correct_final = (preds_final == y_true)

n_both_correct = np.sum(correct_baseline & correct_final)
n_baseline_only = np.sum(correct_baseline & ~correct_final)   # baseline right, final wrong
n_final_only = np.sum(~correct_baseline & correct_final)       # final right, baseline wrong
n_both_wrong = np.sum(~correct_baseline & ~correct_final)

contingency_table = [[n_both_correct, n_baseline_only],
                      [n_final_only, n_both_wrong]]

print("Contingency table:")
print(f"                      Final correct   Final wrong")
print(f"Baseline correct      {n_both_correct:>13}   {n_baseline_only:>11}")
print(f"Baseline wrong        {n_final_only:>13}   {n_both_wrong:>11}")
print()
print(f"Cases only Attention Pooling got right (baseline missed): {n_final_only}")
print(f"Cases only Baseline got right (attention pooling missed): {n_baseline_only}")
print()

result = mcnemar(contingency_table, exact=(n_baseline_only + n_final_only < 25), correction=True)
print(f"McNemar's test statistic: {result.statistic:.4f}")
print(f"p-value: {result.pvalue:.4f}")
print()

alpha = 0.05
if result.pvalue < alpha:
    print(f"p < {alpha} -> The difference between the two models IS statistically significant.")
    if n_final_only > n_baseline_only:
        print("-> Attention Pooling is significantly better than the CLS baseline.")
    else:
        print("-> CLS baseline is significantly better than Attention Pooling.")
else:
    print(f"p >= {alpha} -> The difference is NOT statistically significant.")
    print("-> The overall accuracy gap is consistent with random variation on this dataset.")